# Comparing Scenarios on premise Data

`TimexLCASettings` holds everything one calculation needs - the demand, the
method, the background selection, and every timeline/LCI/LCIA option - so one
object is also the record of what was run. `TimexLCA.from_settings(...).run()`
executes it, `run()` can be called again with overrides, and
`TimexLCA.compare()` runs a list of them into one table.

This notebook runs all of that on a real prospective background: ecoinvent
3.12 (cutoff) plus REMIND-EU SSP2-NDC databases for 2020-2100, built with
[premise](https://github.com/polca/premise), and a small electric-vehicle
foreground - the same production system as this repo's other premise
notebooks.

> The outputs below come from the author's own project (`ei312_REMIND_EU`).
> Re-running this notebook needs an equivalent project of your own; the next
> section shows how to get one.

## Getting the background databases

`bw_timex` itself ships no data. If your project doesn't hold the prospective
databases yet, [PR #222](https://github.com/brightway-lca/bw_timex/pull/222)
builds them for you: `scenario` describes what you want, `create_missing=True`
builds whatever is missing with premise, importing ecoinvent first if needed.

Not executed here - it needs an ecoinvent licence, a premise key, tens of
minutes, and a few GB per year:

```python
tlca = TimexLCA(
    demand={("foreground", "driving"): 1},
    method=("ecoinvent-3.12", "EF v3.1", "climate change", "global warming potential (GWP100)"),
    scenario={
        "iam_model": "remind",
        "pathway": "SSP2-NDC",
        "system_model": "cutoff",
        "ecoinvent_version": "3.12",
        "years": [2020, 2030, 2040, 2050, 2075, 2100],
    },
    create_missing=True,
    premise_key="dummy_premise_decryption_key",              # or $PREMISE_KEY
    ecoinvent_credentials=("dummy_user", "dummy_password"),  # or $ECOINVENT_USERNAME / _PASSWORD
)
```

Databases premise writes carry their own `representative_time` and scenario
metadata (premise >= 2.4.9.2), so nothing has to be mapped by hand
afterwards - and a second run of the same code builds nothing, because the
databases are already there.

## Setting up

The project already holds those databases. The only thing `bw_timex` still
needs is to know the foreground is dynamic - everything else it reads from
the databases' metadata:

In [1]:
from datetime import datetime

import bw2data as bd

from bw_timex import TimexLCA, TimexLCASettings, set_database_metadata

bd.projects.set_current("ei312_REMIND_EU")
set_database_metadata("foreground", representative_time="dynamic")

demand = {("foreground", "driving"): 1}  # one EV over its lifetime
method = ("ecoinvent-3.12", "EF v3.1", "climate change", "global warming potential (GWP100)")

## One calculation: `TimexLCASettings` and `run()`

In [2]:
settings = TimexLCASettings(
    demand=demand,
    method=method,
    starting_datetime=datetime(2020, 6, 1),  # bought in 2020, driven from then on
    build_dynamic_biosphere=False,           # only the static score is needed here
    dynamic_lcia_enabled=False,
    label="bought 2020",
)

tlca = TimexLCA.from_settings(settings).run()
print("base score:  ", round(tlca.base_score))
print("static score:", round(tlca.static_score))

2026-08-23 21:18:08.804 | INFO     | bw_timex.timex_lca:__init__:275 - Initializing TimexLCA object...


2026-08-23 21:18:08.807 | INFO     | bw_timex.timex_lca:__init__:305 - Calculating base LCA...


/Users/timodiepers/Documents/Coding/bw_timex/.venv/lib/python3.12/site-packages/scikits/umfpack/umfpack.py:737: UmfpackWarning: (almost) singular matrix! (estimated cond. number: 3.90e+13)
  warnings.warn(msg, UmfpackWarning)
2026-08-23 21:18:09.520 | INFO     | bw_timex.timex_lca:__init__:322 - Collecting node infos...


2026-08-23 21:18:09.598 | INFO     | bw_timex.timex_lca:__init__:334 - Loading node metadata from 10 database(s)...


2026-08-23 21:18:19.202 | INFO     | bw_timex.timex_lca:__init__:371 - TimexLCA initialized.


...

2026-08-23 21:18:33.724 | INFO     | bw_timex.timex_lca:lci:1139 - Calculating dynamic inventory...


/Users/timodiepers/Documents/Coding/bw_timex/.venv/lib/python3.12/site-packages/scikits/umfpack/umfpack.py:737: UmfpackWarning: (almost) singular matrix! (estimated cond. number: 1.65e+13)
  warnings.warn(msg, UmfpackWarning)
2026-08-23 21:18:35.697 | INFO     | bw_timex.timex_lca:run:560 - Step 3/4: Calculating static LCIA...


2026-08-23 21:18:35.700 | INFO     | bw_timex.timex_lca:run:578 - Step 4/4: Skipping dynamic LCIA (disabled).


2026-08-23 21:18:35.700 | INFO     | bw_timex.timex_lca:run:580 - TimexLCA.run() completed successfully.


base score:   23247
static score: 25463


The time-explicit score is higher than the base score because the EV's
30,000 kWh are spread over its lifetime and resolved against the REMIND-EU
vintage each year of use falls into, rather than the 2020 database the
exchange nominally points at.

Two warnings above are expected: the traversal's score-coverage number only
measures the *foreground* graph it walks, while most of this system's impact
sits in the background, which is resolved by solving the expanded matrix
instead (the score is unchanged across a wide range of `cutoff` values). And
three EV-part productions only have vintages up to 2040 in this project, so
their temporal market falls back to the closest one they have.

`run()` again on the same object, changing only what should change -
the settings object itself is left untouched, and the base LCA is reused:

In [3]:
base_lca_id = id(tlca.base_lca)

tlca.run(starting_datetime=datetime(2075, 6, 1))  # same EV, bought decades later
print("static score, bought 2075:", round(tlca.static_score))
print("base LCA reused:", id(tlca.base_lca) == base_lca_id)

2026-08-23 21:18:35.704 | INFO     | bw_timex.timex_lca:run:529 - Starting TimexLCA.run() pipeline...


2026-08-23 21:18:35.704 | INFO     | bw_timex.timex_lca:run:536 - Step 1/4: Building timeline...


2026-08-23 21:18:35.704 | INFO     | bw_timex.timex_lca:build_timeline:938 - No edge filter function provided. Skipping all edges in background databases.


2026-08-23 21:18:35.705 | INFO     | bw_timex.timex_lca:build_timeline:959 - Creating activity time mapping...


2026-08-23 21:18:35.863 | INFO     | bw_timex.timeline_builder:__init__:113 - Traversing supply chain graph...


...

2026-08-23 21:18:39.654 | INFO     | bw_timex.timex_lca:lci:1139 - Calculating dynamic inventory...


/Users/timodiepers/Documents/Coding/bw_timex/.venv/lib/python3.12/site-packages/scikits/umfpack/umfpack.py:737: UmfpackWarning: (almost) singular matrix! (estimated cond. number: 2.10e+13)
  warnings.warn(msg, UmfpackWarning)
2026-08-23 21:18:42.388 | INFO     | bw_timex.timex_lca:run:560 - Step 3/4: Calculating static LCIA...


2026-08-23 21:18:42.391 | INFO     | bw_timex.timex_lca:run:578 - Step 4/4: Skipping dynamic LCIA (disabled).


2026-08-23 21:18:42.391 | INFO     | bw_timex.timex_lca:run:580 - TimexLCA.run() completed successfully.


static score, bought 2075: 8375
base LCA reused: True


A third of the 2020 purchase's impact, on the same EV - that is REMIND-EU's grid decarbonizing under the vehicle.

## Several calculations: `compare()`

`compare()` takes a list of settings and returns a `ComparisonResult`
whose `summary` holds one row each - the scores next to every setting that
produced them, so the table is its own record of what was run. It builds one
`TimexLCA` per distinct background, so the purchase years below share a
single object:

In [4]:
from dataclasses import replace

comparison = TimexLCA.compare(
    [
        replace(settings, starting_datetime=datetime(year, 6, 1), label=f"bought {year}")
        for year in (2020, 2040, 2075)
    ]
)

comparison.summary[["label", "base_score", "static_score", "timeline_rows", "runtime_s"]]

2026-08-23 21:18:42.395 | INFO     | bw_timex.timex_lca:__init__:275 - Initializing TimexLCA object...


2026-08-23 21:18:42.397 | INFO     | bw_timex.timex_lca:__init__:305 - Calculating base LCA...


/Users/timodiepers/Documents/Coding/bw_timex/.venv/lib/python3.12/site-packages/scikits/umfpack/umfpack.py:737: UmfpackWarning: (almost) singular matrix! (estimated cond. number: 3.90e+13)
  warnings.warn(msg, UmfpackWarning)
2026-08-23 21:18:43.071 | INFO     | bw_timex.timex_lca:__init__:322 - Collecting node infos...


2026-08-23 21:18:43.106 | INFO     | bw_timex.timex_lca:__init__:334 - Loading node metadata from 10 database(s)...


2026-08-23 21:18:43.232 | INFO     | bw_timex.timex_lca:__init__:371 - TimexLCA initialized.


...

2026-08-23 21:19:11.217 | INFO     | bw_timex.timex_lca:lci:1139 - Calculating dynamic inventory...


/Users/timodiepers/Documents/Coding/bw_timex/.venv/lib/python3.12/site-packages/scikits/umfpack/umfpack.py:737: UmfpackWarning: (almost) singular matrix! (estimated cond. number: 2.10e+13)
  warnings.warn(msg, UmfpackWarning)
2026-08-23 21:19:13.877 | INFO     | bw_timex.timex_lca:run:560 - Step 3/4: Calculating static LCIA...


2026-08-23 21:19:13.880 | INFO     | bw_timex.timex_lca:run:578 - Step 4/4: Skipping dynamic LCIA (disabled).


2026-08-23 21:19:13.880 | INFO     | bw_timex.timex_lca:run:580 - TimexLCA.run() completed successfully.


,label,base_score,static_score,timeline_rows,runtime_s
0,bought 2020,23247.151124,25462.997441,27,16.499644
1,bought 2040,23247.151124,9127.954269,27,7.395545
2,bought 2075,23247.151124,8375.067004,27,6.751829


Two options worth knowing: `keep_objects=True` keeps each `TimexLCA`
in `ComparisonResult.objects`, to dig into one result's timeline or dynamic
inventory afterwards; `on_error="record"` puts a failure in the row's `error`
column and carries on, instead of aborting a long unattended sweep.

Comparing *scenarios* rather than purchase years works the same way - give
each settings object a different `scenario={...}` (say `SSP2-NDC` against
`SSP2-PkBudg500`). Each distinct background gets its own `TimexLCA`, since the
background fixes the columns of the time-explicit matrices; that is also why
`run()` refuses a `scenario` change on an existing object and points here
instead.